# 쿼리 모듈 v4 — 단계별 실험 노트북

각 단계를 **따로 불러** 파라미터를 바꿔 가며 확인한다. API(`POST /api/v1/rag/v4/analyze`)와 같은 함수를 쓴다.

| 단계 | 함수 | 바꿔 볼 파라미터 (`configs/query_v4.yaml`) |
|---|---|---|
| ⓪ CSV 값 사전 | `build_index`, `load_index` | `scope` |
| 1-1 재작성(v3, HCX 1회) | `rewrite_v3` | — |
| 1-2 동의어 | `Lexicon.resolve / suggest / scan` | `csv_scope`, `contain_min_len`, `fuzzy_cutoff` |
| 1-3 접지 | `ground_slots` | — |
| 1-4 시점 규칙·되묻기 | `apply_time_rules`, `check`, `apply_selections` | `base_year`, `trend_years`, `n_options` |
| 1-5 서브쿼리 | `decompose` | `split_when_no_common_table` |
| 전체 | `analyze` (+ `Tracer`) | `overrides={...}` |

> 저장소 루트에서 실행. HCX 없이 해 보려면 첫 셀의 `USE_FAKE = True` (가짜 데이터 + HCX 대역).

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parents[1]   # notebooks/v4 에서 열어도
sys.path.insert(0, str(ROOT))
USE_FAKE = os.getenv("V4_NB_FAKE", "0") == "1"      # True: tests/fake_kosis 의 가짜 데이터 + HCX 대역
if USE_FAKE:
    import tempfile
    from tests import fake_kosis
    tmp = Path(tempfile.mkdtemp()) / "storage"
    fake_kosis.build(tmp / "kosis_data")
    for k in ("KOSIS_DATA_DIR", "KOSIS_DATA", "VIEW_OUT_DIR", "TABLE_INDEX_DIR", "VIEW_EMBED_CACHE", "CSV_INDEX_DIR"):
        os.environ[k] = ""
    os.environ["STORAGE_DIR"], os.environ["HCX_API_KEY"] = str(tmp), "test"
    import backend.core.llm as llm
    llm._post = fake_kosis.fake_hcx_post
from backend.core.config import CSV_INDEX_DIR, KOSIS_DATA_DIR
print("원본:", KOSIS_DATA_DIR, "| 값 사전:", CSV_INDEX_DIR)

## ⓪ CSV 값 사전 — 원본 CSV 의 열 머리·분류 값·항목·단위·기간·주석

In [ ]:
from backend.rag.csv_index import build_index, load_index
summary = build_index(scope="all", workers=4)          # 처음 한 번 (이후엔 바뀐 표만 다시 스캔)
idx = load_index()
print({k: summary[k] for k in ("n_tables", "n_values", "by_slot", "n_errors")})

In [ ]:
import pandas as pd
display(pd.DataFrame(idx.values("지역", scope="all")[:10]))
t = idx.table(next(iter(idx.tables)))
print(t["tbl_nm"], "| 열:", [d["obj_nm"] + "(" + d["kind"] + ")" for d in t["dims"]], "| 항목:", [i["name"] for i in t["items"]][:8])
print("주석:", t["footnotes"][:3])

## 설정 — 이 셀에서 값을 바꿔 아래 단계를 다시 실행

In [ ]:
from backend.core.settings_v4 import with_overrides
S = with_overrides({"csv_scope": "all", "base_year": 2026, "trend_years": 5})
S.model_dump()

## 1-2 동의어 조회 — 어느 사전에서 왔는지(source)와 CSV 근거

In [ ]:
from backend.rag.query_analyzer.v4.lexicon import Lexicon
lex = Lexicon(idx, S)
for text, slot in [("출생아 수", "지표"), ("신생아 수", "지표"), ("대구", "지역"), ("인구수", "지표"), ("출산아수", "지표")]:
    hits = lex.resolve(text, slot)
    h = hits[0] if hits else None
    print(f"{text:8s} → {h.value if h else '-':10s} {h.source if h else '':9s} 표 {h.n_tables if h else 0}  "
          f"제안 {lex.suggest(text, slot)[:3] if not (h and h.grounded) else ''}")

## 1-1 재작성 (v3, HCX 1회) → 1-3 접지

In [ ]:
from backend.rag.query_analyzer.v4 import rewrite_v3
from backend.rag.query_analyzer.v4.grounding import ground_slots
Q = "2025년 대전 출생아 수와 사망자 수 차이"
state = rewrite_v3(Q, "2026-11-02")
print("재작성:", state["bundle"]["rewritten"], "| 슬롯:", state["raw_slots"])
grounded, notes = ground_slots(state["raw_slots"], lex)
pd.DataFrame([{"slot": k, "raw": v["raw"], "value": v["value"], "source": v["source"], "match": v["match"], "n_tables": v["n_tables"]}
              for k, vs in grounded.items() for v in vs])

## 1-4 시점 규칙 · 필수 슬롯 · 되묻기 후보

In [ ]:
from backend.rag.query_analyzer.v4.clarify import apply_time_rules, check
for q in ["요즘 대전 출생아 수", "대전 출생아 수 변화", "최근 3년 대전 출생아 수"]:
    raw, rules = apply_time_rules({}, q, [], S)
    print(f"{q:20s} → {raw.get('시점')} {rules}")
g2, _ = ground_slots({"지표": ["출생아수"]}, lex)
for c in check(g2, {"지표": ["출생아수"]}, lex, S):
    print(c["slot"], "|", c["reason"], "|", [o["value"] for o in c["options"]][:10])

## 1-5 서브쿼리 분리

In [ ]:
from backend.rag.query_analyzer.v4.decompose import decompose
slots = {"시점": ["2025년"], "지역": ["대전광역시"], "지표": ["출생아수", "혼인건수"]}
g3, _ = ground_slots(slots, lex)
subs, reason = decompose(g3, slots, [], ["비교"], [], lex, S)
print(reason); pd.DataFrame(subs)

## 전체 — analyze + 단계별 trace (params·출력·초)

In [ ]:
from backend.core.tracking import Tracer
from backend.rag.query_analyzer.v4 import analyze
tr = Tracer("nb", params=S.model_dump(), save=False)       # save=True 면 storage/runs/<run_id>/ 에 저장
a = analyze("출생아 수 알려줘", "2026-11-02", overrides={"csv_scope": "all"}, tracer=tr)
print(a["status"], [(c["slot"], [o["value"] for o in c["options"]][:5]) for c in a["clarifications"]])
pd.DataFrame([{"step": t.step, "seconds": t.seconds, "params": t.params} for t in tr.traces])

In [ ]:
b = analyze("출생아 수 알려줘", "2026-11-02", overrides={"csv_scope": "all"}, previous=a,
            selections={"지역": ["대전광역시"], "시점": ["2025년"]})      # 되묻기 응답 (HCX 다시 안 부름)
print(b["status"]); json.loads(json.dumps(b["view_bundle"], ensure_ascii=False))